In [140]:
import sys
sys.path.insert(0, 'AstroM3')

import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd
from datetime import datetime
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from src_dataloader.data_generator_pytorch import OrganizeData, full_class_dictionary
from src_dataloader.data_preprocessor_pytorch import SpectraProcessor

from AstroM3.core.model import Informer
from AstroM3.core.trainer import Trainer

In [121]:
class DataGenerator(torch.utils.data.Dataset):

    def __init__(self, preprocessed_path, df, step, file_list=None, **kwargs):
        super().__init__(**kwargs)
        self.preprocessed_path = preprocessed_path
        self.step = step
        self.df = df

        if file_list is not None:
            self.data_files = file_list
        else:
            self.data_files = [f for f in os.listdir(preprocessed_path) if f.endswith('.npy')]
        
    def __len__(self): 
        return(len(self.data_files))
    
    def __getitem__(self, index):    
        ''' load processed object alerts to get photometry, metadata, images''' 
        file_path = os.path.join(self.preprocessed_path, str(self.data_files[index]))
        sample = np.load(file_path, allow_pickle=True).item()

        photometry = sample['photometry']
        metadata = sample['metadata'].to_numpy()
        images = sample['images']

        # get spectra csv, save wavelengths fluxes
        obj_id_alert = str(self.data_files[index])
        obj_id = obj_id_alert[:12] # only includes ZTFID from 'ZTFID_alerts.npy' 
        spectra_df = SpectraProcessor.read_spectra_csv(obj_id, '/data/dev/ml_skyportal/AJs_Stuff/(aj)data_all')
        spectra_df = spectra_df.astype(float) # to reassure us <3  

        # get label
        obj_df = self.df[self.df['name'] == obj_id]
        obj_label = obj_df[self.step].iloc[0]
        
        # convert photometry, metadata, images, spectra to tensors
        photometry_tensor = torch.tensor(photometry)
        metadata_tensor = torch.tensor(metadata)
        images_tensor = torch.tensor(images)
        spectra_tensor = torch.from_numpy(spectra_df.values)
        # convert label to tensor
        target = torch.tensor(obj_label, dtype=torch.int64)

        return photometry_tensor, metadata_tensor, images_tensor, spectra_tensor, target

In [110]:
ALERT_PATH = '/data/dev/ml_skyportal/AJs_Stuff/(aj)data_all'
DATA_PATH = 'spectra_data_ID.csv'
KERNEL_PATH = 'kernel.pkl'
    
TEST_DATA_PATH = 'data_test_redux'
TRAIN_DATA_PATH = 'data_train_redux'

In [111]:
data_train = pd.read_csv('data_train.csv')
data_test = pd.read_csv('data_test.csv')

In [112]:
# leave 10 the most common classes
CLASSES = ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event']
data_train = data_train[data_train['type'].isin(CLASSES)]
data_test = data_test[data_test['type'].isin(CLASSES)]

In [ ]:
data_train['type'].unique()[0]

In [114]:
id2target = {i: CLASSES[i] for i in range(10)}
target2id = {v: k for k, v in id2target.items()}

In [116]:
data_train['type_encoded'] = data_train['type'].map(target2id)

In [115]:
data_train['type'].value_counts()

type
SN Ia                     41084
SN II                      4902
SN IIP                     4643
Cataclysmic                1810
AGN                        1806
SN IIn                     1690
SN Ic                      1085
SN Ib                       919
SN IIb                      843
Tidal Disruption Event      561
Name: count, dtype: int64

In [117]:
# downsample SN Ia
sn_ia = data_train[data_train['type'] == 'SN Ia'].sample(n=4902, random_state=42)
data_train = pd.concat([data_train[data_train['type'] != 'SN Ia'], sn_ia])

train_files, val_files  = OrganizeData.split_and_compute_class_weights(data_train, 'type', verbose=True)

train_dataset = DataGenerator(TRAIN_DATA_PATH, data_train, step='type_encoded', file_list=train_files)
val_dataset = DataGenerator(TRAIN_DATA_PATH, data_train, step='type_encoded', file_list=val_files)

In [126]:
photometry, metadata, images, spectra, target = train_dataset[0]

In [127]:
photometry.shape, metadata.shape, images.shape, spectra.shape

(torch.Size([180, 4]),
 torch.Size([10]),
 torch.Size([63, 63, 3]),
 torch.Size([214, 2]))

In [128]:
len(train_dataset), len(val_dataset)

(18477, 4684)

In [129]:
def collate_fn(data):
    photometry, _, _, _, labels = zip(*data)
    
    labels = torch.tensor(labels, dtype=torch.int64)
    
    photometry = torch.stack(photometry)
    photometry_mask = torch.ones((photometry.size(0), photometry.size(1)))
    
    spectra = torch.zeros((len(data), 10))
    metadata = torch.zeros((len(data), 10))
    images = torch.zeros((len(data), 10))
    
    return photometry, photometry_mask, spectra, metadata, labels

In [130]:
# use collate_fn to load the data in the same order as in AstroM3
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True,
                              num_workers=4, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

In [131]:
batch = next(iter(train_dataloader))

In [132]:
photometry, photometry_mask, spectra, metadata, labels = batch
photometry.shape, photometry_mask.shape, spectra.shape, metadata.shape, labels.shape,\
photometry.dtype, photometry_mask.dtype, spectra.dtype, metadata.dtype, labels.dtype

(torch.Size([128, 180, 4]),
 torch.Size([128, 180]),
 torch.Size([128, 10]),
 torch.Size([128, 10]),
 torch.Size([128]),
 torch.float32,
 torch.float32,
 torch.float32,
 torch.float32,
 torch.int64)

In [138]:
config = {
    'project': 'Transients',
    'mode': 'photo',    # 'clip' 'photo' 'spectra' 'meta' 'all'
    'config_from': None,    # 'meridk/AstroCLIPResults/d2u52yml',
    'random_seed': 42,  # 42, 66, 0, 12, 123
    'use_wandb': False,
    'save_weights': False,
    'weights_path': f'/weights/{datetime.now().strftime("%Y-%m-%d-%H-%M")}',
    'use_pretrain': None,
    'freeze': False,
    'num_classes': len(CLASSES),

    # Photometry Model
    'seq_len': 180,
    'p_enc_in': 4,
    'p_d_model': 128,
    'p_dropout': 0.2,
    'p_factor': 1,
    'p_output_attention': False,
    'p_n_heads': 4,
    'p_d_ff': 512,
    'p_activation': 'gelu',
    'p_e_layers': 8,

    # Training
    'batch_size': 512,
    'lr': 0.001,
    'beta1': 0.9,
    'beta2': 0.999,
    'weight_decay': 0.01,
    'epochs': 50,
    'early_stopping_patience': 5,
    'factor': 0.3,  # for ReduceLROnPlateau scheduler
    'patience': 3,  # for ReduceLROnPlateau scheduler
    'warmup': False,
    'warmup_epochs': 0,
    'clip_grad': False,
    'clip_value': 5
}

In [134]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using', device)

model = Informer(config)
model = model.to(device)

Using cuda


In [135]:
with torch.no_grad():
    photometry, photometry_mask = photometry.to(device), photometry_mask.to(device)
    output = model(photometry, photometry_mask)

In [142]:
optimizer = Adam(model.parameters(), lr=config['lr'], betas=(config['beta1'], config['beta2']),
                 weight_decay=config['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=config['factor'], patience=config['patience'])
criterion = torch.nn.CrossEntropyLoss()

trainer = Trainer(model=model, optimizer=optimizer, scheduler=scheduler, warmup_scheduler=warmup_scheduler,
                  criterion=criterion, device=device, config=config)

In [143]:
trainer.train(train_dataloader, val_dataloader, epochs=config['epochs'])

100%|████████████████████████████████████████████████████████████████████████████| 37/37 [00:15<00:00,  2.31it/s]


Epoch 0: Train Loss 4.8235 	 Val Loss 2.2175 	                     Train Acc 0.1814 	 Val Acc 0.2009


100%|████████████████████████████████████████████████████████████████████████████| 37/37 [00:14<00:00,  2.62it/s]


Epoch 1: Train Loss 2.2101 	 Val Loss 2.2066 	                     Train Acc 0.1953 	 Val Acc 0.2146


100%|████████████████████████████████████████████████████████████████████████████| 37/37 [00:14<00:00,  2.58it/s]


...

100%|████████████████████████████████████████████████████████████████████████████| 37/37 [00:15<00:00,  2.35it/s]


Epoch 29: Train Loss 1.3696 	 Val Loss 1.4925 	                     Train Acc 0.5216 	 Val Acc 0.4878


100%|████████████████████████████████████████████████████████████████████████████| 37/37 [00:14<00:00,  2.48it/s]

Epoch 30: Train Loss 1.3295 	 Val Loss 1.4854 	                     Train Acc 0.5361 	 Val Acc 0.4863
Early stopping at epoch 30


1.4777289612873181